In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()




# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df_Q3 = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df_Q3.head()

In [ ]:
# Task 3: Write your code here:
df_Q3.info()

In [ ]:
# Task 4: Write your code here:
df_Q3.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df_Q3.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_Q3)
df_Q3 = df_Q3.fillna(df_Q3.mean())
check_missing_values(df_Q3)


In [ ]:
# Task 2: Write your code here:
duplicates = df_Q3.duplicated().sum()

df_Q3.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

categorical_cols = df_Q3.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
   le = LabelEncoder()
   df_Q3[col] = le.fit_transform(df_Q3[col].astype(str))

   df_Q3.head()

In [ ]:
# Task 4: Write your code here:
num_cols = df_Q3.select_dtypes(include=["number"]).columns
print("num Columns:", list(num_cols))

scaler = StandardScaler()

# TODO: Apply fit_transform to scale the numerical columns
df_Q3[num_cols] = scaler.fit_transform(df_Q3[num_cols])

df_Q3.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df_Q3, target_column):
    print("Target Distribution:")
    print(df_Q3[target_column].value_counts(normalize=True))
    sns.countplot(x=df_Q3[target_column])
    plt.title("Target Distribution")
    plt.show()


In [ ]:
# Task 1: Write your code here:
X = df_Q3.drop("Target",axis=1)

y = df_Q3["Target"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sklearn_models = {
  "K-Nearest Neighbors": KNeighborsClassifier(
      n_neighbors=3,
  ),
  "Support Vector Machine": SVC(
      kernel='rbf',
      C=0.75
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=3
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=320,
      max_depth=4
  ),
  "XGBoost": XGBClassifier(
      verbosity=0,
      n_estimators=300,
      max_depth=5,
      learning_rate=0.05
  ),
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  ) }


In [ ]:
all_results = {}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)


In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
  print(f"  Precision: {np.mean(all_results[model_name]['precision']):.4f}")
  print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here: importances = {}
importance = pd.DataFrame({
    'feature': num_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: